# 🛡️ Federated Learning for IoT Network Intrusion Detection (FL-IoT-IDS)
## 🚀 Google Colab Multi-Scenario Experiment Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/colab_experiment_runner.ipynb)

This notebook provides an automated, end-to-end experimental harness to pull the repository from GitHub and run all intrusion detection benchmark scenarios on Google Colab:
* **Unified Hyperparameter Configuration:** Single declarative `ExperimentConfig` defining all data, architecture, and federated simulation settings.
* **Scenario E1:** Centralized PyTorch Baseline (Upper Bound Performance)
* **Scenario E2:** Federated Learning IID Baseline (FedAvg)
* **Scenario E4:** Moderate Statistical Heterogeneity (Dirichlet $\alpha = 0.5$)
* **Scenario E5:** Severe Statistical Heterogeneity (Dirichlet $\alpha = 0.1$ with FedProx $\mu$ Regularization)
* **Comparative Evaluation:** Macro-F1, Minority Recall (Web-based & Brute-force attacks), and Normalized Confusion Matrices.

---
### 🛠️ Step 1: Environment Setup & GitHub Repository Synchronization
We check GPU acceleration, clone the repository `https://github.com/NVKQ2022/FedLearning.git`, and install dependencies.

In [ ]:
# Check hardware acceleration
import torch
print(f'PyTorch Version: {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')


In [ ]:
# Clone repository from GitHub (or pull latest updates)
import os, sys, subprocess
repo_url = 'https://github.com/NVKQ2022/FedLearning.git'
repo_dir = '/content/FedLearning'

if not os.path.exists(repo_dir):
    print(f'Cloning {repo_url} to {repo_dir}...')
    subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)
else:
    print('Repository already present. Pulling latest updates from origin/main...')
    subprocess.run(['git', '-C', repo_dir, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', repo_dir, 'reset', '--hard', 'origin/main'], check=True)

# Navigate to repository root
os.chdir(repo_dir)

# Add repository root to sys.path to enable clean imports from src
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# Purge any cached in-memory src modules so fresh code is loaded
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

print('Current Working Directory:', os.getcwd())


In [ ]:
# Install required dependencies
!pip install -q -r requirements.txt
print('Dependencies installed successfully.')


---
### ⚙️ Step 2: Global Hyperparameter & Scenario Configuration
**Edit any hyperparameters here before running any experiments.** All benchmark scenarios consume settings directly from this single configuration container.

In [ ]:
from src.utils.config import ExperimentConfig
from src.utils.seed import set_seed

# ==============================================================================
# 🛠️ EDIT HYPERPARAMETERS HERE (Single Source of Truth)
# ==============================================================================
CONFIG = ExperimentConfig(
    experiment_name="colab_multi_scenario_benchmark",
    seed=42,                            # Integer seed for 100% bit-level reproducibility
    
    # --- Dataset & Sampling ---
    sample_size=50000,                  # Set 50000 for rapid execution, or None for full 5.11M rows
    test_size=0.2,                      # 20% holdout test set
    val_size=0.1,                       # 10% validation set
    scaler_type="robust",               # 'robust' (median/IQR) or 'standard'
    batch_size=128,                     # Mini-batch size
    
    # --- Model Architecture ---
    hidden_dims=(128, 64),              # Default (128, 64) or deeper funnel (256, 128, 64)
    dropout_rate=0.2,                   # Dropout probability
    
    # --- Loss & Imbalance Strategy ---
    loss_type="focal_loss",             # 'focal_loss', 'cross_entropy', or 'weighted_ce'
    class_weight_strategy="balanced",   # 'balanced' (inverse frequency), 'sqrt_balanced', or 'none'
    focal_gamma=2.0,                    # Focusing parameter
    
    # --- Centralized Training ---
    learning_rate=1e-3,
    weight_decay=1e-4,
    epochs=10,                          # Max epochs for Scenario E1
    patience=3,
    monitor_metric="val_loss",          # 'val_loss' (lowest loss) or 'val_acc' (highest accuracy)
    
    # --- Federated Learning Simulations (E2, E4, E5) ---
    num_clients=5,                      # Total participating edge clients
    num_rounds=8,                       # Server aggregation communication rounds
    local_epochs=2,                     # Client local training epochs per round
    mu=0.05,                            # FedProx proximal coefficient (Scenario E5)
    dirichlet_alpha=0.1                 # Severe Non-IID Dirichlet skew (Scenario E5)
)

# Enforce global deterministic seed
set_seed(CONFIG.seed)

# Print formatted hyperparameter overview
CONFIG.display()


---
### 💾 Step 3: Dataset Acquisition (Google Drive or Synthetic Fallback)
The raw dataset `merged_CICIOT2023_data.csv` is ~991 MB and excluded from Git.
You can either:
1. Mount Google Drive if you uploaded the CSV there (`/content/drive/MyDrive/NguyenVietKyQuanKLTN/...`).
2. Or allow the automatic fallback generator below to synthesize a realistic CICIoT2023-compatible dataset for instant execution.

In [ ]:
# 1. Mount Google Drive to access real dataset
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except Exception as e:
    print('Drive mount skipped or not running on Colab:', e)

import os, shutil
import pandas as pd, numpy as np

target_csv_path = CONFIG.target_csv_path
os.makedirs(os.path.dirname(target_csv_path), exist_ok=True)

gdrive_csv = CONFIG.gdrive_csv
if not os.path.exists(gdrive_csv):
    if os.path.exists(CONFIG.alt_gdrive_csv):
        gdrive_csv = CONFIG.alt_gdrive_csv

# 2. Check if real dataset exists in Google Drive
if os.path.exists(gdrive_csv):
    gdrive_size = os.path.getsize(gdrive_csv) / (1024**2)
    if not os.path.exists(target_csv_path) or os.path.getsize(target_csv_path) < 100 * 1024 * 1024:
        print(f'Found REAL dataset in Google Drive ({gdrive_size:.2f} MB)! Copying to local Colab SSD...')
        shutil.copy(gdrive_csv, target_csv_path)
        print(f'✅ Real dataset copied successfully: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB')
    else:
        print(f'✅ Real dataset already present on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB')
elif not os.path.exists(target_csv_path):
    print('⚠️ Real dataset NOT found at', gdrive_csv)
    print('Please verify that merged_CICIOT2023_data.csv exists under NguyenVietKyQuanKLTN/datasets/ in Google Drive!')
    print('Generating synthetic CICIoT2023-compatible dataset (21 MB) for quick pipeline dry-run...')
    n_samples = 30000
    classes = ['Benign', 'DDoS', 'DoS', 'Recon', 'Spoof', 'Mirai', 'Web-based', 'Brute-force']
    p = [0.18, 0.38, 0.23, 0.10, 0.05, 0.04, 0.015, 0.005]
    
    mock_data = {f'feat_{i}': np.random.exponential(scale=1.5, size=n_samples) for i in range(39)}
    mock_data['feat_0'] = np.random.pareto(a=1.5, size=n_samples) * 10.0
    mock_data['group_class_name'] = np.random.choice(classes, size=n_samples, p=p)
    
    pd.DataFrame(mock_data).to_csv(target_csv_path, index=False)
    print(f'⚠️ Synthetic dataset created ({os.path.getsize(target_csv_path)/(1024**2):.2f} MB). Note: Features are random noise!')
else:
    current_size = os.path.getsize(target_csv_path) / (1024**2)
    if current_size < 100:
        print(f'⚠️ Warning: Currently using SYNTHETIC dry-run dataset ({current_size:.2f} MB, random noise).')
        print('   Upload real merged_CICIOT2023_data.csv (~991 MB) to Google Drive and re-run this cell to train with real data!')
    else:
        print(f'✅ Real CICIoT2023 dataset ready: {current_size:.2f} MB')


---
### 🔄 Step 4: Leak-Free Preprocessing & Class Weights
We use `load_and_preprocess_ciciot2023` to clean infinities, impute medians, apply $\log(1+x)$ compression, scale with `RobustScaler`, and compute class weights.

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights

# Load and preprocess with stratified holdout test split (70/10/20)
data = load_and_preprocess_ciciot2023(
    csv_path=CONFIG.target_csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(
    y=data['y_train'],
    num_classes=data['num_classes'],
    strategy=CONFIG.class_weight_strategy
)
class_names = data['class_names']
minority_classes = ['Web-based', 'Brute-force']

print(f'Train samples: {len(data["X_train"]):,}')
print(f'Train loader batches: {len(data["train_loader"]):,}')
print(f'Validation samples: {len(data["X_val"]):,}')
print(f'Holdout Test samples: {len(data["X_test"]):,}')
print('Class Names:', class_names)
print(f'Computed Class Weights ({CONFIG.class_weight_strategy}):', np.round(class_weights, 3))


---
### 🏆 Step 5: Scenario E1 - Centralized Baseline (Upper Bound)
We train a centralized `TabularIoTMLP` with `MultiClassFocalLoss` and early stopping on validation loss.

In [ ]:
from src.models.mlp import TabularIoTMLP
from src.losses.focal_loss import build_loss_function
from src.optimizers.optimizer import build_optimizer
from src.training.trainer import CentralizedTrainer
from src.evaluation.evaluator import evaluate_comprehensive

# 1. Instantiate model, loss, optimizer
e1_model = TabularIoTMLP(
    input_dim=data['input_dim'],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data['num_classes'],
    dropout_rate=CONFIG.dropout_rate
).to(device)
criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights if CONFIG.class_weight_strategy != 'none' else None,
    gamma=CONFIG.focal_gamma,
    device=device
)
optimizer = build_optimizer(
    model=e1_model,
    optimizer_type=CONFIG.optimizer_type,
    lr=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay
)

# 2. Fit with CentralizedTrainer
trainer_e1 = CentralizedTrainer(e1_model, optimizer, criterion, device=device, max_grad_norm=CONFIG.max_grad_norm)
history_e1 = trainer_e1.fit(
    train_loader=data['train_loader'],
    val_loader=data['val_loader'],
    epochs=CONFIG.epochs,
    patience=CONFIG.patience,
    monitor=CONFIG.monitor_metric,
    verbose=CONFIG.verbose
)

# 3. Comprehensive holdout evaluation
e1_results = evaluate_comprehensive(
    model=e1_model,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print(f'Scenario E1 Centralized Test Macro-F1: {e1_results["macro_f1"]*100:.2f}%')
print(f'Overall Accuracy: {e1_results["accuracy"]*100:.2f}%')
print('Minority Recall:', e1_results['minority_recall'])


---
### 🌐 Step 6: Scenario E2 - Federated IID Baseline (FedAvg)
Data is partitioned uniformly across $K$ edge clients using Stratified IID partitioning.

In [ ]:
from src.data.partition import partition_iid, create_client_dataloaders
from src.training.trainer import LocalClientTrainer
import copy

num_clients = CONFIG.num_clients
num_rounds = CONFIG.num_rounds
local_epochs = CONFIG.local_epochs

iid_parts = partition_iid(data['y_train'], num_clients=num_clients, seed=CONFIG.seed)
client_loaders_iid = create_client_dataloaders(data['X_train'], data['y_train'], iid_parts, batch_size=64)

global_model_e2 = TabularIoTMLP(
    input_dim=data['input_dim'],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data['num_classes'],
    dropout_rate=CONFIG.dropout_rate
).to(device)

print(f'Starting Scenario E2: {num_clients} clients, {num_rounds} rounds, {local_epochs} local epochs...')
for round_idx in range(1, num_rounds + 1):
    client_weights = []
    for client_id in range(num_clients):
        local_m = copy.deepcopy(global_model_e2)
        opt = build_optimizer(local_m, CONFIG.optimizer_type, lr=CONFIG.learning_rate)
        c_trainer = LocalClientTrainer(local_m, opt, criterion, device=device)
        c_trainer.train_epochs(client_loaders_iid[client_id], num_epochs=local_epochs, mu=0.0)
        client_weights.append(local_m.get_weights())
    
    # FedAvg Server Aggregation
    avg_w = [np.mean([cw[l] for cw in client_weights], axis=0) for l in range(len(client_weights[0]))]
    global_model_e2.set_weights(avg_w)
    print(f'  [Round {round_idx:02d}/{num_rounds:02d}] Aggregation complete.')

e2_results = evaluate_comprehensive(
    model=global_model_e2,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)
print(f'Scenario E2 IID FedAvg Test Macro-F1: {e2_results["macro_f1"]*100:.2f}%')
print('Minority Recall:', e2_results['minority_recall'])


---
### 🔥 Step 7: Scenario E5 - Severe Non-IID Dirichlet Skew (FedAvg vs. FedProx)
We simulate severe label heterogeneity where rare attacks are concentrated in single edge clients.
We evaluate **FedProx** ($\mu = CONFIG.mu$) to constrain client parameter drift.

In [ ]:
from src.data.partition import partition_dirichlet

# 1. Generate Dirichlet Non-IID partition
non_iid_parts = partition_dirichlet(
    y=data['y_train'],
    num_clients=CONFIG.num_clients,
    alpha=CONFIG.dirichlet_alpha,
    min_samples_per_client=100,
    seed=CONFIG.seed
)
client_loaders_non_iid = create_client_dataloaders(data['X_train'], data['y_train'], non_iid_parts, batch_size=64)

# 2. Train with FedProx
global_model_e5 = TabularIoTMLP(
    input_dim=data['input_dim'],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data['num_classes'],
    dropout_rate=CONFIG.dropout_rate
).to(device)
mu_val = CONFIG.mu

print(f'Starting Scenario E5 (Dirichlet alpha={CONFIG.dirichlet_alpha}, FedProx mu={mu_val})...')
for round_idx in range(1, CONFIG.num_rounds + 1):
    client_weights = []
    for client_id in range(CONFIG.num_clients):
        local_m = copy.deepcopy(global_model_e5)
        opt = build_optimizer(local_m, CONFIG.optimizer_type, lr=CONFIG.learning_rate)
        c_trainer = LocalClientTrainer(local_m, opt, criterion, device=device)
        
        # FedProx proximal penalty: (mu / 2) * ||w - w_t||^2
        c_trainer.train_epochs(
            client_loaders_non_iid[client_id],
            num_epochs=CONFIG.local_epochs,
            global_model=global_model_e5,
            mu=mu_val
        )
        client_weights.append(local_m.get_weights())
    
    avg_w = [np.mean([cw[l] for cw in client_weights], axis=0) for l in range(len(client_weights[0]))]
    global_model_e5.set_weights(avg_w)
    print(f'  [Round {round_idx:02d}/{CONFIG.num_rounds:02d}] Aggregation complete.')

e5_results = evaluate_comprehensive(
    model=global_model_e5,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)
print(f'Scenario E5 Non-IID FedProx Test Macro-F1: {e5_results["macro_f1"]*100:.2f}%')
print('Minority Recall:', e5_results['minority_recall'])


---
### 📊 Step 8: Comparative Analysis & Publication Figures
We plot comparative performance bars and the normalized confusion matrix across scenarios.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Macro-F1 Comparison Bar Chart
scenarios = ['E1: Centralized\n(Upper Bound)', 'E2: Federated IID\n(FedAvg)', f'E5: Non-IID\n(FedProx $\\mu={CONFIG.mu}$)']
macro_scores = [e1_results['macro_f1'] * 100, e2_results['macro_f1'] * 100, e5_results['macro_f1'] * 100]
colors = ['#2ecc71', '#3498db', '#e74c3c']

bars = axes[0].bar(scenarios, macro_scores, color=colors, edgecolor='black', width=0.5)
axes[0].set_ylabel('Macro-F1 Score (%)', fontsize=12)
axes[0].set_ylim(0, 105)
axes[0].set_title('Intrusion Detection Performance Across Scenarios', fontsize=13, fontweight='bold')
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f'{yval:.2f}%', ha='center', va='bottom', fontweight='bold')

# 2. Normalized Confusion Matrix for Scenario E5
cm = e5_results['confusion_matrix']
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[1], cbar=False)
axes[1].set_title('Scenario E5 Normalized Confusion Matrix', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Class', fontsize=12)
axes[1].set_ylabel('True Class', fontsize=12)
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
os.makedirs('reports/figures', exist_ok=True)
plt.savefig('reports/figures/colab_scenario_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


---
### 💾 Step 9: Persist Results, Config & Metrics to Google Drive
Export metrics, figures, and active `ExperimentConfig` to Google Drive to protect against Colab runtime disconnections.

In [ ]:
import json

export_dict = {
    'E1_Centralized': {k: v for k, v in e1_results.items() if k != 'confusion_matrix'},
    'E2_IID_FedAvg': {k: v for k, v in e2_results.items() if k != 'confusion_matrix'},
    'E5_Severe_Non_IID_FedProx': {k: v for k, v in e5_results.items() if k != 'confusion_matrix'},
}

local_metrics_file = 'reports/colab_benchmark_metrics.json'
with open(local_metrics_file, 'w') as f:
    json.dump(export_dict, f, indent=4)

config_file = 'reports/experiment_config.json'
CONFIG.save_json(config_file)

print(f'Metrics saved locally to {local_metrics_file}')
print(f'Config saved locally to {config_file}')

# Optional export to Google Drive if mounted
drive_reports_dir = CONFIG.reports_dir
if not os.path.exists(drive_reports_dir) and os.path.exists('/content/drive/MyDrive/FedLearning'):
    drive_reports_dir = '/content/drive/MyDrive/FedLearning/reports'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_reports_dir, exist_ok=True)
    shutil.copy(local_metrics_file, os.path.join(drive_reports_dir, 'colab_benchmark_metrics.json'))
    shutil.copy(config_file, os.path.join(drive_reports_dir, 'experiment_config.json'))
    shutil.copy('reports/figures/colab_scenario_comparison.png', os.path.join(drive_reports_dir, 'colab_scenario_comparison.png'))
    print(f'Results and plots successfully archived to Google Drive: {drive_reports_dir}')
else:
    print('Google Drive not mounted. Results stored locally in Colab runtime.')
